# Independent Final Submission Notebook

This notebook is self-contained: it reads `data/train.csv` and `data/test.csv`, trains models, and writes `submission.csv` without loading a precomputed prediction file.

Public leaderboard feedback from the previous experiment:

- The previous static anchor submission scored **12.04499**.
- The target-encoding + calibration experiment scored **12.45622**, so that path is removed from the default pipeline.

The default model here is a conservative CatBoost + ExtraTrees ensemble using the same feature families as the research notebook, plus OOF blend-weight selection. It also writes a small set of independent candidate submissions under `submissions/` for Kaggle testing.

In [1]:
from pathlib import Path
import time
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from IPython.display import display
from catboost import CatBoostRegressor
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder

ROOT = Path.cwd()
DATA_DIR = ROOT / "data"
SUBMISSION_DIR = ROOT / "submissions"
SUBMISSION_DIR.mkdir(exist_ok=True)

TARGET = "property_organic_content"
RANDOM_STATE = 42
N_FOLDS = 5

# Public feedback rejected the target-encoding experiment, so keep this conservative.
SEEDS = [42, 2025]
DOCUMENTED_CATBOOST_WEIGHT = 0.475

CATBOOST_PARAMS = {
    "iterations": 1000,
    "depth": 6,
    "learning_rate": 0.045,
    "l2_leaf_reg": 8,
    "random_strength": 1.2,
    "loss_function": "RMSE",
    "eval_metric": "RMSE",
    "early_stopping_rounds": 130,
    "verbose": False,
    "allow_writing_files": False,
    "thread_count": -1,
}

EXTRATREES_PARAMS = {
    "n_estimators": 600,
    "min_samples_leaf": 2,
    "max_features": 0.8,
    "n_jobs": -1,
}

pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.5f}")

## 1. Load Data

In [2]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")
y = train[TARGET].astype(float).copy()

assert TARGET in train.columns
assert TARGET not in test.columns
assert train["sample_id"].is_unique
assert test["sample_id"].is_unique
assert list(sample_submission.columns) == ["sample_id", TARGET]
assert sample_submission["sample_id"].equals(test["sample_id"])

print(f"Train shape: {train.shape}")
print(f"Test shape : {test.shape}")
print(f"Target mean/std: {y.mean():.5f} / {y.std():.5f}")
print(f"Unseen test source_id count: {len(set(test['source_id']) - set(train['source_id']))}")

Train shape: (11210, 52)
Test shape : (2670, 51)
Target mean/std: 34.14252 / 23.19861
Unseen test source_id count: 0


## 2. Feature Engineering

In [3]:
def rmse(y_true, y_pred):
    return root_mean_squared_error(y_true, y_pred)


def regression_metrics(y_true, y_pred):
    return {
        "RMSE": rmse(y_true, y_pred),
        "MAE": mean_absolute_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred),
    }


def make_features(df):
    X = df.drop(columns=[TARGET, "sample_id"], errors="ignore").copy()

    band_a = [c for c in X.columns if c.startswith("spectral_band_A_PC_")]
    band_b = [c for c in X.columns if c.startswith("spectral_band_B_PC_")]
    chemistry = [
        "property_acidity_index", "cation_Ca", "cation_Mg",
        "cation_Na", "cation_exchange_capacity",
    ]

    X["missing_total"] = X.isna().sum(axis=1)
    X["chem_missing_count"] = X[chemistry].isna().sum(axis=1)
    X["band_B_available_actual"] = np.where(X[band_b].notna().any(axis=1), "YES", "NO")
    X["band_B_missing_count"] = X[band_b].isna().sum(axis=1)
    X["coordinates_available"] = np.where(X[["latitude", "longitude"]].notna().all(axis=1), "YES", "NO")

    for prefix, cols in [("A", band_a), ("B", band_b)]:
        values = X[cols]
        X[f"spectral_{prefix}_mean"] = values.mean(axis=1)
        X[f"spectral_{prefix}_std"] = values.std(axis=1)
        X[f"spectral_{prefix}_l2"] = np.sqrt((values ** 2).sum(axis=1))
        X[f"spectral_{prefix}_max_abs"] = values.abs().max(axis=1)
        X[f"spectral_{prefix}_abs_sum"] = values.abs().sum(axis=1)
        X[f"spectral_{prefix}_pos_count"] = (values > 0).sum(axis=1)

    eps = 1e-6
    X["particle_total"] = X["property_particle_coarse"] + X["property_particle_fine"]
    X["fine_fraction"] = X["property_particle_fine"] / (X["particle_total"].abs() + eps)
    X["coarse_fine_ratio"] = X["property_particle_coarse"] / (X["property_particle_fine"].abs() + eps)
    X["base_cation_sum"] = X[["cation_Ca", "cation_Mg", "cation_Na"]].sum(axis=1, min_count=1)
    X["ca_mg_ratio"] = X["cation_Ca"] / (X["cation_Mg"].abs() + eps)
    X["base_to_cec_ratio"] = X["base_cation_sum"] / (X["cation_exchange_capacity"].abs() + eps)
    X["log_cec"] = np.log1p(X["cation_exchange_capacity"].clip(lower=0))

    X["geo_hierarchy"] = (
        X["geo_zone_macro"].astype(str) + "|"
        + X["geo_zone_meso"].astype(str) + "|"
        + X["geo_zone_micro"].astype(str)
    )
    X["biome_landcover"] = X["biome"].astype(str) + "|" + X["land_cover_type"].astype(str)
    X["source_bandB"] = X["source_id"].astype(str) + "|" + X["band_B_available_actual"].astype(str)
    X["macro_biome"] = X["geo_zone_macro"].astype(str) + "|" + X["biome"].astype(str)
    X["macro_landcover"] = X["geo_zone_macro"].astype(str) + "|" + X["land_cover_type"].astype(str)

    X["lat_round_1"] = X["latitude"].round(1)
    X["lon_round_1"] = X["longitude"].round(1)

    categorical_cols = X.select_dtypes(include=["object", "string", "category"]).columns.tolist()
    for col in categorical_cols:
        X[col] = X[col].fillna("Missing").astype(str)
    return X


def make_extratrees_pipeline(numeric_cols, categorical_cols, seed):
    preprocessor = ColumnTransformer([
        ("numeric", SimpleImputer(strategy="median"), numeric_cols),
        ("categorical", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), categorical_cols),
    ])
    model = ExtraTreesRegressor(random_state=seed, **EXTRATREES_PARAMS)
    return Pipeline([("preprocess", preprocessor), ("model", model)])


def clip_predictions(pred):
    return np.clip(np.asarray(pred, dtype=float), 0, y.max())


X = make_features(train)
X_test = make_features(test).reindex(columns=X.columns)

numeric_cols = X.select_dtypes(include=np.number).columns.tolist()
categorical_cols = [c for c in X.columns if c not in numeric_cols]

print(f"Engineered feature shape: train={X.shape}, test={X_test.shape}")
print(f"Numeric features: {len(numeric_cols)} | categorical features: {len(categorical_cols)}")

Engineered feature shape: train=(11210, 81), test=(2670, 81)
Numeric features: 63 | categorical features: 18


## 3. Train Seed Ensemble

This is intentionally conservative: no target encoding and no automatic calibration. The failed public-LB experiment showed that aggressive local OOF improvements can generalize poorly.

In [4]:
cat_oofs = []
et_oofs = []
cat_tests = []
et_tests = []
fold_rows = []

for seed in SEEDS:
    started_seed = time.perf_counter()
    cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    bins = pd.qcut(y, q=10, labels=False, duplicates="drop")

    cat_oof_seed = np.zeros(len(train))
    et_oof_seed = np.zeros(len(train))
    cat_test_folds = []
    et_test_folds = []

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, bins), start=1):
        fold_start = time.perf_counter()
        X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
        y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

        cat_model = CatBoostRegressor(random_seed=seed + fold, **CATBOOST_PARAMS)
        cat_model.fit(
            X_train,
            y_train,
            cat_features=categorical_cols,
            eval_set=(X_valid, y_valid),
            verbose=False,
        )

        et_model = make_extratrees_pipeline(numeric_cols, categorical_cols, seed + fold)
        et_model.fit(X_train, y_train)

        cat_valid = cat_model.predict(X_valid)
        et_valid = et_model.predict(X_valid)
        cat_test = cat_model.predict(X_test)
        et_test = et_model.predict(X_test)

        cat_oof_seed[valid_idx] = cat_valid
        et_oof_seed[valid_idx] = et_valid
        cat_test_folds.append(cat_test)
        et_test_folds.append(et_test)

        elapsed = time.perf_counter() - fold_start
        fold_rows.append({
            "seed": seed,
            "fold": fold,
            "catboost_RMSE": rmse(y_valid, cat_valid),
            "extratrees_RMSE": rmse(y_valid, et_valid),
            "seconds": elapsed,
        })
        print(
            f"seed={seed} fold={fold}: "
            f"cat={rmse(y_valid, cat_valid):.5f}, "
            f"et={rmse(y_valid, et_valid):.5f}, "
            f"seconds={elapsed:.1f}"
        )

    cat_oofs.append(cat_oof_seed)
    et_oofs.append(et_oof_seed)
    cat_tests.append(np.mean(cat_test_folds, axis=0))
    et_tests.append(np.mean(et_test_folds, axis=0))
    print(f"Seed {seed} finished in {(time.perf_counter() - started_seed) / 60:.1f} minutes")

cat_oof = np.mean(cat_oofs, axis=0)
et_oof = np.mean(et_oofs, axis=0)
cat_test_pred = np.mean(cat_tests, axis=0)
et_test_pred = np.mean(et_tests, axis=0)

fold_metrics = pd.DataFrame(fold_rows)
model_summary = pd.DataFrame([
    {"model": "CatBoost seed ensemble", **regression_metrics(y, cat_oof)},
    {"model": "ExtraTrees seed ensemble", **regression_metrics(y, et_oof)},
]).set_index("model")

display(fold_metrics)
display(model_summary)

seed=42 fold=1: cat=11.95677, et=11.97190, seconds=159.8


seed=42 fold=2: cat=11.73767, et=11.78362, seconds=158.9


seed=42 fold=3: cat=11.95383, et=11.58115, seconds=60.0


seed=42 fold=4: cat=11.87544, et=11.99686, seconds=39.8


seed=42 fold=5: cat=10.94664, et=10.93486, seconds=39.4
Seed 42 finished in 7.6 minutes


...

seed=2025 fold=3: cat=12.14957, et=11.85918, seconds=38.8


seed=2025 fold=4: cat=11.22402, et=11.21120, seconds=39.4


seed=2025 fold=5: cat=11.98575, et=12.21871, seconds=39.8
Seed 2025 finished in 3.3 minutes


,seed,fold,catboost_RMSE,extratrees_RMSE,seconds
0,42,1,11.95677,11.97190,159.82736
1,42,2,11.73767,11.78362,158.87660
2,42,3,11.95383,11.58115,60.00604
3,42,4,11.87544,11.99686,39.79412
4,42,5,10.94664,10.93486,39.36052
5,2025,1,11.37867,11.62029,39.26535
6,2025,2,11.29384,11.18789,38.66118
7,2025,3,12.14957,11.85918,38.82494
8,2025,4,11.22402,11.21120,39.44379
9,2025,5,11.98575,12.21871,39.77951


,RMSE,MAE,R2
model,,,
CatBoost seed ensemble,11.57122,7.31159,0.75119
ExtraTrees seed ensemble,11.57309,7.24427,0.75111


## 4. Blend Search and Candidate Files

In [5]:
def make_blend(cat_weight):
    et_weight = 1 - cat_weight
    return (
        cat_weight * cat_oof + et_weight * et_oof,
        cat_weight * cat_test_pred + et_weight * et_test_pred,
    )


blend_rows = []
for cat_weight in np.arange(0.20, 0.801, 0.025):
    pred, _ = make_blend(float(cat_weight))
    blend_rows.append({
        "catboost_weight": round(float(cat_weight), 3),
        "extratrees_weight": round(float(1 - cat_weight), 3),
        **regression_metrics(y, pred),
    })

blend_search = pd.DataFrame(blend_rows).sort_values("RMSE").reset_index(drop=True)
best_oof_cat_weight = float(blend_search.loc[0, "catboost_weight"])

print(f"Best OOF blend: {best_oof_cat_weight:.3f} CatBoost / {1 - best_oof_cat_weight:.3f} ExtraTrees")
display(blend_search.head(12))

Best OOF blend: 0.500 CatBoost / 0.500 ExtraTrees


,catboost_weight,extratrees_weight,RMSE,MAE,R2
0,0.50000,0.50000,11.26202,7.07119,0.76431
1,0.52500,0.47500,11.26276,7.07329,0.76428
2,0.47500,0.52500,11.26285,7.07009,0.76427
3,0.55000,0.45000,11.26507,7.07646,0.76418
4,0.45000,0.55000,11.26526,7.07010,0.76417
5,0.57500,0.42500,11.26895,7.08072,0.76402
6,0.42500,0.57500,11.26924,7.07115,0.76400
7,0.60000,0.40000,11.27440,7.08586,0.76379
8,0.40000,0.60000,11.27478,7.07345,0.76377
9,0.62500,0.37500,11.28141,7.09231,0.76349


In [6]:
def write_submission(name, pred):
    output = sample_submission.copy()
    output[TARGET] = clip_predictions(pred)
    path = SUBMISSION_DIR / f"{name}.csv"
    output.to_csv(path, index=False)
    return path, output


def candidate_row(name, cat_weight, notes):
    oof_pred, test_pred = make_blend(cat_weight)
    path, output = write_submission(name, test_pred)
    return {
        "name": name,
        "path": str(path),
        "catboost_weight": cat_weight,
        "extratrees_weight": 1 - cat_weight,
        "notes": notes,
        **regression_metrics(y, clip_predictions(oof_pred)),
        "pred_mean": output[TARGET].mean(),
        "pred_std": output[TARGET].std(),
        "pred_p95": output[TARGET].quantile(0.95),
        "pred_p99": output[TARGET].quantile(0.99),
    }

candidate_rows = []
candidate_rows.append(candidate_row(
    "submission_independent_documented_weight",
    DOCUMENTED_CATBOOST_WEIGHT,
    "documented conservative blend from research notebook",
))
candidate_rows.append(candidate_row(
    "submission_independent_oof_best_weight",
    best_oof_cat_weight,
    "best local OOF blend weight",
))

for cat_weight in [0.40, 0.50, 0.60]:
    candidate_rows.append(candidate_row(
        f"submission_independent_weight_{int(cat_weight * 100):02d}",
        cat_weight,
        "nearby public-LB probe weight",
    ))

candidate_summary = pd.DataFrame(candidate_rows).sort_values("RMSE").reset_index(drop=True)
display(candidate_summary)

# The root submission must be independent. Use the OOF-best conservative blend, not a precomputed anchor.
final_path = Path(candidate_summary.loc[0, "path"])
final_submission = pd.read_csv(final_path)
final_submission.to_csv(ROOT / "submission.csv", index=False)
print(f"Root submission.csv copied from independent candidate: {final_path.name}")
print(f"Best independent local OOF RMSE: {candidate_summary.loc[0, 'RMSE']:.5f}")

,name,path,catboost_weight,extratrees_weight,notes,RMSE,MAE,R2,pred_mean,pred_std,pred_p95,pred_p99
0,submission_independent_oof_best_weight,C:\Users\nicho\Documents\Projects\compfest\sub...,0.50000,0.50000,best local OOF blend weight,11.26202,7.07119,0.76431,34.59837,19.42646,74.89060,104.22135
1,submission_independent_weight_50,C:\Users\nicho\Documents\Projects\compfest\sub...,0.50000,0.50000,nearby public-LB probe weight,11.26202,7.07119,0.76431,34.59837,19.42646,74.89060,104.22135
2,submission_independent_documented_weight,C:\Users\nicho\Documents\Projects\compfest\sub...,0.47500,0.52500,documented conservative blend from research no...,11.26285,7.07009,0.76427,34.60147,19.41786,74.71250,104.30677
3,submission_independent_weight_60,C:\Users\nicho\Documents\Projects\compfest\sub...,0.60000,0.40000,nearby public-LB probe weight,11.27440,7.08586,0.76379,34.58597,19.46789,74.74823,103.62226
4,submission_independent_weight_40,C:\Users\nicho\Documents\Projects\compfest\sub...,0.40000,0.60000,nearby public-LB probe weight,11.27478,7.07345,0.76377,34.61077,19.39630,74.83301,103.91787


Root submission.csv copied from independent candidate: submission_independent_oof_best_weight.csv
Best independent local OOF RMSE: 11.26202


## 5. Final Validation

In [7]:
check = pd.read_csv(ROOT / "submission.csv")
assert list(check.columns) == list(sample_submission.columns)
assert len(check) == len(sample_submission)
assert check["sample_id"].equals(sample_submission["sample_id"])
assert check[TARGET].notna().all()
assert np.isfinite(check[TARGET]).all()

print("submission.csv validation passed")
display(check[TARGET].describe(percentiles=[0.01, 0.05, 0.50, 0.95, 0.99]).to_frame("submission"))

submission.csv validation passed


,submission
count,"2,670.00000"
mean,34.59837
std,19.42646
min,8.03858
1%,12.25908
5%,14.91273
50%,28.34885
95%,74.89060
99%,104.22135
max,141.91058
